In [1]:
# Linear Regression exercises using GA-Brain dataset from Machine Learning for Biomedical Applications with Scikit-Learn and
# Pytorch by Deprez & Robinson
# Goal: predict GA (gestational age) from various brain statistics including whole brain volume
# File created: 25 November 2026
# Last edited: 25 November 2026

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from scipy.stats import norm

np.set_printoptions(suppress = True)
import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("Datasets/GA-brain-volumes-1-feature.csv", header = None, names = ["GA", "BrainVol"])
data.head()

,GA,BrainVol
0,35.714,252.41
1,37.429,277.73
2,36.143,266.36
3,36.714,266.13
4,42.286,308.35


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 164 entries, 0 to 163
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   GA        164 non-null    float64
 1   BrainVol  164 non-null    float64
dtypes: float64(2)
memory usage: 2.7 KB


In [5]:
# Predicting GA from BrainVol using sklearn.linear_model's LinearRegression - ML for Biomedical Apps by Deprez & Robinson
print("Simple Linear Regression: GA = b0 + b1 x BrainVol + eps")
Y = data["GA"].to_numpy()
X = data["BrainVol"].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = LinearRegression()
model.fit(X, Y)
beta0 = model.intercept_
beta1 = model.coef_.item()
print("beta0 = %s, beta1 = %s" % (beta0, beta1))
Rsquare = model.score(X, Y)
print("Rsquare = %s" % Rsquare)

# Section 3.1.4 Exercises Q5 asks for RMSE on whole dataset and cross-validated RMSE
Yhat = model.predict(X)
RMSE = np.sqrt((np.linalg.norm(Yhat - Y) ** 2) / len(Y))
print("RMSE = %s" % RMSE)

test_scores = cross_validate(LinearRegression(), X, Y, scoring = "neg_root_mean_squared_error")["test_score"]
print("Cross validation RMSE = %s" % -np.mean(test_scores))

Simple Linear Regression: GA = b0 + b1 x BrainVol + eps
(164, 1) (164,)
beta0 = 24.817988846429305, beta1 = 0.04324525431266588
Rsquare = 0.8455356622591337
RMSE = 1.5639380824529276
Cross validation RMSE = 1.5979898631743867


In [6]:
class Jie_SimpleLinearRegression:
    
    def __init__(self):
        self.beta0 = None
        self.beta1 = None
        self.sigma2 = None
        self.Rsquare = None
        self.beta0_stdev = None
        self.beta1_stdev = None
        self.beta0_t = None
        self.beta1_t = None
        self.beta0_p = None
        self.beta1_p = None
        self.results = None
    
    def fit(self, X, Y, predictor_name):
        
        Xc = np.insert(X, 0, np.ones(len(X)), axis = 1)
        self.beta0, self.beta1 = np.linalg.inv(Xc.T @ Xc) @ Xc.T @ Y
        
        Yhat = self.predict(X)
        self.sigma2 = (np.linalg.norm(Y - Yhat) ** 2) / (len(Y) - 2)
        
        beta_var = np.diag(self.sigma2 * np.linalg.inv(Xc.T @ Xc))
        self.beta0_stdev, self.beta1_stdev = np.sqrt(beta_var)

        self.beta0_t = self.beta0 / self.beta0_stdev
        self.beta1_t = self.beta1 / self.beta1_stdev
        
        self.beta0_p = 2 * norm.sf(np.abs(self.beta0_t))
        self.beta1_p = 2 * norm.sf(np.abs(self.beta1_t))

        beta = np.array([self.beta0, self.beta1])
        beta_stdev = np.array([self.beta0_stdev, self.beta1_stdev])
        beta_t = np.array([self.beta0_t, self.beta1_t])
        beta_p = np.array([self.beta0_p, self.beta1_p])
        self.results = pd.DataFrame(zip(beta, beta_stdev, beta_t, beta_p),
                                    columns = ["beta", "beta stdev", "beta t", "beta p"],
                                    index = ["intercept", predictor_name])
        print(self.results)
        
    def score(self, X, Y):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = self.predict(X)
        RSS = np.linalg.norm(Y - Yhat) ** 2
        TSS = np.linalg.norm(Y - np.mean(Y)) ** 2
        self.Rsquare = 1 - RSS/TSS
        print("Rsquare = %s" % self.Rsquare)
        
    def predict(self, X):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = (self.beta0 + self.beta1 * X).reshape(-1)
        return Yhat

In [7]:
# Predicting GA from BrainVol by hand - ML for Biomedical Apps by Deprez & Robinson
print("Simple Linear Regression: GA = b0 + b1 x BrainVol + eps")
Y = data["GA"].to_numpy()
predictor_name = "BrainVol"
X = data[predictor_name].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = Jie_SimpleLinearRegression()
model.fit(X, Y, predictor_name)
model.score(X, Y)

# Section 3.1.4 Exercises Q5 asks for RMSE on whole dataset and cross-validated RMSE
Yhat = model.predict(X)
RMSE = np.sqrt((np.linalg.norm(Yhat - Y) ** 2) / len(Y))
print("RMSE = %s" % RMSE)

# test_scores = cross_validate(Jie_SimpleLinearRegression(), X, Y, scoring = "neg_root_mean_squared_error")["test_score"]
# print("Cross validation RMSE = %s" % -np.mean(test_scores))

Simple Linear Regression: GA = b0 + b1 x BrainVol + eps
(164, 1) (164,)
                beta  beta stdev     beta t         beta p
intercept  24.817989    0.428174  57.962455   0.000000e+00
BrainVol    0.043245    0.001452  29.778949  7.319109e-195
Rsquare = 0.8455356622591337
RMSE = 1.5639380824529274
